#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
# @title ⚙️ 1. Install Dependencies & Clone Repository
# @markdown Run this cell to set up the system. It installs the `uv` package manager and clones the Index-TTS repository.

import os
from IPython.display import clear_output

print("🚀 Cloning Index-TTS repository...")
!git clone https://github.com/index-tts/index-tts.git

# Move into the directory for all future commands
%cd /content/index-tts

print("📦 Installing uv package manager...")
!pip install -U uv

print("🛠️ Installing all dependencies (this may take a few minutes)...")
!uv sync --all-extras

clear_output()
print("✅ Setup complete! Environment is ready.")

In [ ]:
# @title 📥 2. Download Index-TTS 2.5 Models
# @markdown This cell downloads the core IndexTTS-2.5 model weights needed for production generation.

from IPython.display import clear_output

print("🧠 Downloading IndexTTS-2.5 model weights...")
!uv tool install "huggingface-hub"
!uv run huggingface-cli download IndexTeam/IndexTTS-2.5 --local-dir=checkpoints

clear_output()
print("✅ Models downloaded and ready for production!")

In [ ]:
# @title 🗣️ 3. Voice Cloning (Single Reference Audio)
# @markdown Enter your text and select the language. When you run the cell, you will be prompted to upload your reference audio.

text_to_speak = "Enter the script you want the AI to generate here." # @param {type:"string"}
language = "EN" # @param ["EN", "ZH", "JP", "ES", "AR"]
output_filename = "output_clone_IndexTTS-2.5-CoinNoin.wav" # @param {type:"string"}

import os
from google.colab import files
from IPython.display import Audio, display, clear_output

print("📤 Please upload your reference voice audio file (.wav):")
uploaded = files.upload()
reference_audio_path = list(uploaded.keys())[0]
clear_output()
print(f"✅ Using reference audio: {reference_audio_path}")
print("⏳ Generating voice... Please wait.")

script = f"""
from indextts.infer_v2_5 import IndexTTS2

# Initialize model
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)

# Run inference
tts.infer(
    spk_audio_prompt='{reference_audio_path}',
    text='{text_to_speak}',
    lang='{language}',
    output_path="{output_filename}",
    verbose=False
)
"""

with open("run_clone.py", "w") as f:
    f.write(script)

!uv run python run_clone.py > /dev/null 2>&1

clear_output()
if os.path.exists(output_filename):
    print("✅ Generation Complete!")
    print("🎧 Result:")
    display(Audio(output_filename))
else:
    print("❌ Error: Audio generation failed.")

In [ ]:
# @title 🎭 4. Emotion Control (Audio Reference)
# @markdown Run this cell to upload TWO files: your base speaker voice, and the emotion reference voice. Adjust `emo_alpha` to control emotion intensity.

text_to_speak = "Enter your emotional script here." # @param {type:"string"}
language = "EN" # @param ["EN", "ZH", "JP", "ES", "AR"]
emotion_intensity = 0.9 # @param {type:"slider", min:0.0, max:1.0, step:0.1}
output_filename = "output_emo_audio_IndexTTS-2.5-CoinNoin.wav" # @param {type:"string"}

import os
from google.colab import files
from IPython.display import Audio, display, clear_output

print("📤 1. Please upload your BASE SPEAKER audio file (.wav):")
uploaded_spk = files.upload()
speaker_voice_path = list(uploaded_spk.keys())[0]

print("\n📤 2. Please upload your EMOTION REFERENCE audio file (.wav):")
uploaded_emo = files.upload()
emotion_audio_path = list(uploaded_emo.keys())[0]

clear_output()
print(f"✅ Base voice: {speaker_voice_path} | Emotion reference: {emotion_audio_path}")
print("⏳ Generating voice... Please wait.")

script = f"""
from indextts.infer_v2_5 import IndexTTS2
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)

tts.infer(
    spk_audio_prompt='{speaker_voice_path}',
    text="{text_to_speak}",
    lang="{language}",
    emo_audio_prompt="{emotion_audio_path}",
    emo_alpha={emotion_intensity},
    output_path="{output_filename}",
    verbose=False
)
"""

with open("run_emo_audio.py", "w") as f:
    f.write(script)

!uv run python run_emo_audio.py > /dev/null 2>&1

clear_output()
if os.path.exists(output_filename):
    print("✅ Generation Complete!")
    display(Audio(output_filename))
else:
    print("❌ Error: Audio generation failed.")

In [ ]:
# @title 📊 5. Emotion Control via Vector Parameters
# @markdown Define emotion precisely by mixing levels of: [happy, angry, sad, afraid, disgusted, melancholic, surprised, calm]. You will be prompted to upload the base voice.

text_to_speak = "Enter the script you want to apply emotion vectors to." # @param {type:"string"}
language = "EN" # @param ["EN", "ZH", "JP", "ES", "AR"]
happy = 0.8 # @param {type:"slider", min:0.0, max:1.0, step:0.1}
angry = 0.0 # @param {type:"slider", min:0.0, max:1.0, step:0.1}
sad = 0.0 # @param {type:"slider", min:0.0, max:1.0, step:0.1}
output_filename = "output_emo_vector_IndexTTS-2.5-CoinNoin.wav" # @param {type:"string"}

import os
from google.colab import files
from IPython.display import Audio, display, clear_output

print("📤 Please upload your reference voice audio file (.wav):")
uploaded = files.upload()
reference_voice_path = list(uploaded.keys())[0]

clear_output()
print(f"✅ Using reference audio: {reference_voice_path}")
print("⏳ Generating voice... Please wait.")

script = f"""
from indextts.infer_v2_5 import IndexTTS2
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)

# 8 floats: [happy, angry, sad, afraid, disgusted, melancholic, surprised, calm]
emo_vec = [{happy}, {angry}, {sad}, 0, 0, 0, 0, 0]

tts.infer(
    spk_audio_prompt='{reference_voice_path}',
    text="{text_to_speak}",
    lang="{language}",
    emo_vector=emo_vec,
    use_random=False,
    output_path="{output_filename}",
    verbose=False
)
"""

with open("run_emo_vector.py", "w") as f:
    f.write(script)

!uv run python run_emo_vector.py > /dev/null 2>&1

clear_output()
if os.path.exists(output_filename):
    print("✅ Generation Complete!")
    display(Audio(output_filename))
else:
    print("❌ Error: Audio generation failed.")

In [ ]:
# @title 🤖 6. Automatic Emotion Detection from Text
# @markdown The AI will automatically guess the correct emotion from your text. (Note: Loads an extra Qwen model into memory). You will be prompted to upload the base voice.

text_to_speak = "Enter a highly emotional sentence, like: Watch out! It's right behind you!" # @param {type:"string"}
language = "EN" # @param ["EN", "ZH", "JP", "ES", "AR"]
emotion_intensity = 0.6 # @param {type:"slider", min:0.0, max:1.0, step:0.1}
output_filename = "output_emo_auto_IndexTTS-2.5-CoinNoin.wav" # @param {type:"string"}

import os
from google.colab import files
from IPython.display import Audio, display, clear_output

print("📤 Please upload your reference voice audio file (.wav):")
uploaded = files.upload()
reference_voice_path = list(uploaded.keys())[0]

clear_output()
print(f"✅ Using reference audio: {reference_voice_path}")
print("⏳ Generating voice... Please wait.")

script = f"""
from indextts.infer_v2_5 import IndexTTS2

# Important: use_qwen_emo=True is required for text-based emotion detection
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True, use_qwen_emo=True)

tts.infer(
    spk_audio_prompt='{reference_voice_path}',
    text="{text_to_speak}",
    lang="{language}",
    emo_alpha={emotion_intensity},
    use_emo_text=True,
    use_random=False,
    output_path="{output_filename}",
    verbose=False
)
"""

with open("run_emo_auto.py", "w") as f:
    f.write(script)

!uv run python run_emo_auto.py > /dev/null 2>&1

clear_output()
if os.path.exists(output_filename):
    print("✅ Generation Complete!")
    display(Audio(output_filename))
else:
    print("❌ Error: Audio generation failed.")

In [ ]:
# @title 📝 7. Control Emotion with an Explicit Description
# @markdown Tell the AI exactly how the speaker feels in the `emotion_description` field. You will be prompted to upload the base voice.

text_to_speak = "I can't believe we actually won the championship!" # @param {type:"string"}
emotion_description = "Describe the feeling: Overwhelming tears of joy and disbelief." # @param {type:"string"}
language = "EN" # @param ["EN", "ZH", "JP", "ES", "AR"]
emotion_intensity = 0.6 # @param {type:"slider", min:0.0, max:1.0, step:0.1}
output_filename = "output_emo_desc_IndexTTS-2.5-CoinNoin.wav" # @param {type:"string"}

import os
from google.colab import files
from IPython.display import Audio, display, clear_output

print("📤 Please upload your reference voice audio file (.wav):")
uploaded = files.upload()
reference_voice_path = list(uploaded.keys())[0]

clear_output()
print(f"✅ Using reference audio: {reference_voice_path}")
print("⏳ Generating voice... Please wait.")

script = f"""
from indextts.infer_v2_5 import IndexTTS2
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True, use_qwen_emo=True)

tts.infer(
    spk_audio_prompt='{reference_voice_path}',
    text="{text_to_speak}",
    lang="{language}",
    emo_alpha={emotion_intensity},
    use_emo_text=True,
    emo_text="{emotion_description}",
    use_random=False,
    output_path="{output_filename}",
    verbose=False
)
"""

with open("run_emo_desc.py", "w") as f:
    f.write(script)

!uv run python run_emo_desc.py > /dev/null 2>&1

clear_output()
if os.path.exists(output_filename):
    print("✅ Generation Complete!")
    display(Audio(output_filename))
else:
    print("❌ Error: Audio generation failed.")

In [ ]:
# @title ⏱️ 8. Speaking Speed Control
# @markdown Adjust the `speed_factor`. 1.0 is normal. >1.0 slows down the speech. <1.0 speeds it up. You will be prompted to upload the base voice.

text_to_speak = "Enter the script you want to speed up or slow down." # @param {type:"string"}
language = "EN" # @param ["EN", "ZH", "JP", "ES", "AR"]
speed_factor = 1.0 # @param {type:"slider", min:0.5, max:2.0, step:0.1}
output_filename = "output_speed_IndexTTS-2.5-CoinNoin.wav" # @param {type:"string"}

import os
from google.colab import files
from IPython.display import Audio, display, clear_output

print("📤 Please upload your reference voice audio file (.wav):")
uploaded = files.upload()
reference_voice_path = list(uploaded.keys())[0]

clear_output()
print(f"✅ Using reference audio: {reference_voice_path}")
print("⏳ Generating voice... Please wait.")

script = f"""
from indextts.infer_v2_5 import IndexTTS2
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)

tts.infer(
    spk_audio_prompt='{reference_voice_path}',
    text="{text_to_speak}",
    lang="{language}",
    duration_factor={speed_factor},
    output_path="{output_filename}",
    verbose=False
)
"""

with open("run_speed.py", "w") as f:
    f.write(script)

!uv run python run_speed.py > /dev/null 2>&1

clear_output()
if os.path.exists(output_filename):
    print("✅ Generation Complete!")
    display(Audio(output_filename))
else:
    print("❌ Error: Audio generation failed.")

In [ ]:
# @title 🔤 9. Exact Pronunciation Control (CMU Phonemes)
# @markdown Force the AI to pronounce words specifically using `<word|PHONEMES>`. Check CMU dictionary for English phonetic spelling rules.

text_to_speak = "He had a <minute|M IH1 . N AH0 T> to examine the <minute|M AY0 . N UW1 T> details." # @param {type:"string"}
language = "EN" # @param ["EN", "ZH", "JP", "ES", "AR"]
output_filename = "output_pronunciation_IndexTTS-2.5-CoinNoin.wav" # @param {type:"string"}

import os
from google.colab import files
from IPython.display import Audio, display, clear_output

print("📤 Please upload your reference voice audio file (.wav):")
uploaded = files.upload()
reference_voice_path = list(uploaded.keys())[0]

clear_output()
print(f"✅ Using reference audio: {reference_voice_path}")
print("⏳ Generating voice... Please wait.")

script = f"""
from indextts.infer_v2_5 import IndexTTS2
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints", use_bf16=True)

tts.infer(
    spk_audio_prompt='{reference_voice_path}',
    text="{text_to_speak}",
    lang="{language}",
    output_path="{output_filename}",
    verbose=False
)
"""

with open("run_pronunciation.py", "w") as f:
    f.write(script)

!uv run python run_pronunciation.py > /dev/null 2>&1

clear_output()
if os.path.exists(output_filename):
    print("✅ Generation Complete!")
    display(Audio(output_filename))
else:
    print("❌ Error: Audio generation failed.")